In [ ]:
# import necessary libraries
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt

In [ ]:
# code execution
palette_20_ = [
    '#EB545C',
    '#5084C2',
    '#DBA091',
    '#D7EF9B',
    '#EF7512',
    '#289E92',
    '#878787',
    '#FC6FCF',
    '#F52831',
    '#80FF08',
    '#CC66FF',
    '#FFFF0A',
    '#AABBCC',
    '#DDEEFF',
    '#FF9900',
    '#0099FF',
    '#FF00FF',
    '#F0CE58',
    '#B487B7',
    '#7F7F7F']

import matplotlib.colors as mcolors

palette_20 = mcolors.ListedColormap(palette_20_)

In [ ]:
### functions: 0. preprocess the data 1. compute UMI
### 2. use corrected counts to compute umi and pie chart
### 3. identify highly variable genes for Leiden clusters and annotate cell types
### 4. Gsea enrichment for host genes

import gffutils
import os
import math
from adjustText import adjust_text
import harmonypy as hm

def preprocessing(folder_path, image_path, res = 160, mode = 'tsv'):
    if mode=='csv':
        adata = sc.read_csv(folder_path+'/correcteddata.csv').T
        get_spatialmeta(adata,folder_path,image_path,mode=mode,res=res)
    else:
        adata = sc.read_csv(folder_path+'/filtered_matrix_correct_new.tsv',delimiter=None)
        get_spatialmeta(adata,folder_path,image_path,mode=mode,res=res)
        
    get_varmeta(adata)
    get_umi(adata)
        
    return adata
        
def get_spatialmeta(adata,folder_path,image_path,mode='tsv',res=160):
    if mode=='csv':
        coor = adata.obs_names.str[1:]
    else:
        coor = adata.obs_names
    coor = coor.to_series()
    tmp = coor.str.split('x').values
    spatial = np.zeros((adata.shape[0],2))
    for i in range(spatial.shape[0]):
        spatial[i,0] = int(tmp[i][0])
        spatial[i,1] = int(tmp[i][1])
    adata.obsm['spatial'] = spatial - 0.5
    img = plt.imread(folder_path+'/'+image_path)
    spatial_key = "spatial"
    library_id = "tissue42"
    adata.uns[spatial_key] = {library_id: {}}
    adata.uns[spatial_key][library_id]["images"] = {}
    adata.uns[spatial_key][library_id]["images"] = {"hires": img}
    adata.uns[spatial_key][library_id]["scalefactors"] = {"tissue_hires_scalef": res, "spot_diameter_fullres": 0.9}
    return
    
    
def get_varmeta(adata):
    adata.var['species'] = 'Host'
    adata.var_names = adata.var_names.str.replace('-', '_')
    genes_bac = adata.var_names[np.array([a.count('_RS') for a in adata.var_names])>0]
    adata.var['species'][np.array([a.count('_RS') for a in adata.var_names])>0] = np.array([a.split('_RS')[0] for a in genes_bac])
    from pathlib import Path
    directory = Path('../genome_bacteria') 
    adata.var['biotype'] = 'Host'
    adata.var['start_pos'] = 0
    adata.var['chr'] = 'unknown'
    var_names_set = set(adata.var_names)
    
    for filename in directory.rglob('*.gtf'):
        db_filename = str(filename)[0:-3] + 'db'
        #db = gffutils.create_db(str(filename), dbfn=db_filename,merge_strategy='merge')
        db = gffutils.FeatureDB(db_filename)
        genes = db.features_of_type('gene')
        for gene in genes:
            if gene.id in var_names_set:
                adata.var['biotype'][gene.id] = gene.attributes.get('gene_biotype', ['unknown'])[0]
                adata.var['start_pos'][gene.id] = gene.start
                adata.var['chr'][gene.id] = gene.seqid
                
    binsize = 50000
    
    adata.var['bac_mRNA_bin'] = (adata.var['start_pos']/binsize).astype(int)
    varnames = adata.var_names[(adata.var['species']!='Host') & (adata.var['biotype']=='protein_coding')]
    df = pd.DataFrame(adata[:,(adata.var['species']!='Host') & (adata.var['biotype']=='protein_coding')].X,columns=varnames,index=adata.obs_names)
    df = df.T
    df['bac_mRNA_bin'] = adata.var['bac_mRNA_bin'][(adata.var['species']!='Host') & (adata.var['biotype']=='protein_coding')].values.copy()
    df['species'] = adata.var['species'][(adata.var['species']!='Host') & (adata.var['biotype']=='protein_coding')].values.copy()
    df['chr'] = adata.var['chr'][(adata.var['species']!='Host') & (adata.var['biotype']=='protein_coding')].values.copy()
    df['bac_mRNA_bin'] = df['bac_mRNA_bin'].astype(str)
    df = df.groupby(['species','chr','bac_mRNA_bin']).sum().dropna()
    df = df.T
    df.columns = ['_'.join(col).strip() for col in df.columns.values]
    
    adata.obsm['bac_mRNA_bin'] = df            
    mapping = pd.read_csv('../mapping.csv',index_col=0)
    adata.var['Gram'] = '/'
    adata.var['name'] = 'Host'
    adata.var['name'][adata.var['species']!='Host'] = mapping.loc[adata.var['species'][adata.var['species']!='Host']]['Short name'].values.copy()
    adata.var['Gram'][adata.var['species']!='Host'] = mapping.loc[adata.var['species'][adata.var['species']!='Host']]['Gram'].values.copy()
    return

def get_umi(adata):
    df = pd.DataFrame(index=adata.obs_names)
    for i in adata.var['species'].unique():
        if i != 'Host':
            adata.obs[i+'_mRNA'] = adata[:,(adata.var['biotype']=='protein_coding') & (adata.var_names.str.startswith(i))].X.sum(axis=1).copy()
            adata.obs[i+'_tRNA'] = adata[:,(adata.var['biotype']=='tRNA') & (adata.var_names.str.startswith(i))].X.sum(axis=1).copy()
            adata.obs[i+'_rRNA'] = adata[:,(adata.var['biotype']=='rRNA') & (adata.var_names.str.startswith(i))].X.sum(axis=1).copy()
            df[i] = adata[:,(adata.var_names.str.startswith(i))].X.sum(axis=1).copy()
        else:
            adata.obs[i+'_mRNA'] = adata[:,(adata.var['biotype']=='protein_coding') & (adata.var['species']==i)].X.sum(axis=1).copy()
            adata.obs[i+'_tRNA'] = adata[:,(adata.var['biotype']=='tRNA') & (adata.var['species']==i)].X.sum(axis=1).copy()
            adata.obs[i+'_rRNA'] = adata[:,(adata.var['biotype']=='rRNA') & (adata.var['species']==i)].X.sum(axis=1).copy()
            #df[i] = adata[:,(adata.var_names.str.startswith(i))].X.sum(axis=1).copy()
    
    adata.obs['bac_mRNA'] = adata.obs[[i + '_mRNA' for i in list(adata.var['species'].unique()[1:].astype(str))]].values.sum(axis=1)
    adata.obs['bac_rRNA'] = adata.obs[[i + '_rRNA' for i in list(adata.var['species'].unique()[1:].astype(str))]].values.sum(axis=1)
    adata.obs['host_umi'] = adata[:,adata.var['species']=='Host'].X.sum(axis=1).copy()
    adata.obs['bac_umi'] = adata[:,adata.var['species']!='Host'].X.sum(axis=1).copy()
    adata.obsm['species_abundance'] = df
    return

def vis_comparison(mixpaths,gfpath):
    adata_all = None
    for mixpath in mixpaths:
        adata_sc = sc.read_csv(mixpath+'/sctransform.csv').T
        adata_sc.obs['condition'] = 'BacteriaMix'
        adata_sc.obs['batch'] = mixpath
        if adata_all is None:
            adata_all = adata_sc
        else:
            adata_all = sc.concat([adata_all,adata_sc])
    adata_ct = sc.read_csv(gfpath+'/sctransform.csv').T
    adata_ct.obs['condition'] = 'GF'
    adata_ct.obs['batch'] = 'GF'
    adata_ct = adata_ct[:,~adata_ct.var_names.str.contains('-RS')]
    adata_ct = adata_ct[:,~adata_ct.var_names.str.contains('Gm33838')]
    adata_all = sc.concat([adata_all,adata_ct])
    sc.pp.pca(adata_all,n_comps=10)
    ho = hm.run_harmony(adata_all.obsm['X_pca'], adata_all.obs, 'batch',reference_values="GF")
    # Write the adjusted PCs to a new file.
    adata_all.obsm['corrected_pca'] = ho.Z_corr.T
    
    
    adata_allf = None
    for mixpath in mixpaths:
        #adata_scf = sc.read_csv(mixpath+'/correcteddata2.csv').T
        adata_scf = sc.read_csv(mixpath+'/sctransform.csv').T
        adata_scf.obs['condition'] = 'BacteriaMix'
        adata_scf.obs['batch'] = mixpath
        if adata_allf is None:
            adata_allf = adata_scf
        else:
            adata_allf = sc.concat([adata_allf,adata_scf])
    #adata_ctf = sc.read_csv(gfpath+'/correcteddata2.csv').T
    adata_ctf = sc.read_csv(gfpath+'/sctransform.csv').T
    adata_ctf.obs['condition'] = 'GF'
    adata_ctf.obs['batch'] = 'GF'
    #adata_ctf = adata_ctf[:,~adata_ctf.var_names.str.contains('_RS')]
    adata_allf = sc.concat([adata_allf,adata_ctf])

    adata_allf.obsm['X_pca'] = adata_all.obsm['X_pca'].copy()
    adata_allf.obsm['corrected_pca'] = adata_all.obsm['corrected_pca'].copy()
    sc.pp.neighbors(adata_allf,use_rep='corrected_pca')
    sc.tl.umap(adata_allf)
    #sc.pp.neighbors(adata_allf)
    sc.tl.leiden(adata_allf,resolution=0.35)
    sc.pl.umap(adata_allf,color=['condition','batch','leiden'])
    return adata_allf

def diffgenes(adata,label,es=0.5):
    #os.mkdir('genelist')
    for i in adata.obs[label].unique():
        adata_tmp = adata[adata.obs[label]==i].copy()
        adata_tmp.X[adata_tmp.X<0] = 0
        sc.tl.rank_genes_groups(adata_tmp, 'condition', method='t-test',use_raw=False,reference='GF')
        volcano(adata_tmp,save='figures/'+i)
        a = pd.DataFrame(pd.DataFrame(adata_tmp.uns['rank_genes_groups']['pvals_adj']).iloc[:,0].values,index=pd.DataFrame(adata_tmp.uns['rank_genes_groups']['names']).iloc[:,0])
        a['logfoldchanges'] = pd.DataFrame(adata_tmp.uns['rank_genes_groups']['logfoldchanges']).iloc[:,0].values
        #a['effect_size'] = (adata_tmp[adata_tmp.obs['condition']=='BacteriaMix',a.index].X.mean(axis=0) - adata_tmp[adata_tmp.obs['condition']=='GF',a.index].X.mean(axis=0)).copy()
        a = a.loc[a[0]<0.05]
        a = a.loc[a['logfoldchanges']>es]
        a.to_csv('genelist/'+i+'up.csv')
        a = pd.DataFrame(pd.DataFrame(adata_tmp.uns['rank_genes_groups']['pvals_adj']).iloc[:,0].values,index=pd.DataFrame(adata_tmp.uns['rank_genes_groups']['names']).iloc[:,0])
        a['logfoldchanges'] = pd.DataFrame(adata_tmp.uns['rank_genes_groups']['logfoldchanges']).iloc[:,0].values
        #a['effect_size'] = (-adata_tmp[adata_tmp.obs['condition']=='BacteriaMix',a.index].X.mean(axis=0) + adata_tmp[adata_tmp.obs['condition']=='GF',a.index].X.mean(axis=0)).copy()
        a = a.loc[a[0]<0.05]
        a = a.loc[a['logfoldchanges']<(-es)]
        a.to_csv('genelist/'+i+'down.csv')

def diffgenes2(adata,label,es=0.5):
    #os.mkdir('genelist')
    adata_tmp = adata.copy()
    adata_tmp.X[adata_tmp.X<0] = 0
    sc.tl.rank_genes_groups(adata_tmp, label, use_raw=False)
    volcano(adata_tmp,save='figures/'+label)
    a = pd.DataFrame(pd.DataFrame(adata_tmp.uns['rank_genes_groups']['pvals_adj']).iloc[:,0].values,index=pd.DataFrame(adata_tmp.uns['rank_genes_groups']['names']).iloc[:,0])
    a['logfoldchanges'] = pd.DataFrame(adata_tmp.uns['rank_genes_groups']['logfoldchanges']).iloc[:,0].values
    #a['effect_size'] = (adata_tmp[adata_tmp.obs['condition']=='BacteriaMix',a.index].X.mean(axis=0) - adata_tmp[adata_tmp.obs['condition']=='GF',a.index].X.mean(axis=0)).copy()
    a = a.loc[a[0]<0.05]
    a = a.loc[a['logfoldchanges']>es]
    a.to_csv('genelist/'+label+'up.csv')
    a = pd.DataFrame(pd.DataFrame(adata_tmp.uns['rank_genes_groups']['pvals_adj']).iloc[:,0].values,index=pd.DataFrame(adata_tmp.uns['rank_genes_groups']['names']).iloc[:,0])
    a['logfoldchanges'] = pd.DataFrame(adata_tmp.uns['rank_genes_groups']['logfoldchanges']).iloc[:,0].values
    #a['effect_size'] = (-adata_tmp[adata_tmp.obs['condition']=='BacteriaMix',a.index].X.mean(axis=0) + adata_tmp[adata_tmp.obs['condition']=='GF',a.index].X.mean(axis=0)).copy()
    a = a.loc[a[0]<0.05]
    a = a.loc[a['logfoldchanges']<(-es)]
    a.to_csv('genelist/'+label+'down.csv')
        
import gseapy as gp
def gsea(adata,label):
    df_all = None
    for i in adata.obs[label].cat.categories:
        j = 'up'
        gl = pd.read_csv('genelist/'+i+j+'.csv',index_col=0).index.values
        if gl.shape[0]>0:
            enr = gp.enrichr(gene_list=list(gl), # or "./tests/data/gene_list.txt",
                         gene_sets=['WikiPathways_2019_Mouse'],
                         organism='Mus musculus', # don't forget to set organism to the one you desired! e.g. Yeast
                         outdir=None, # don't write to disk
                        )
            if not enr.results.empty:
                if enr.results['Adjusted P-value'].min() < 0.05:
                    dotplot(enr.results,column = 'Adjusted P-value',size=50,cmap='viridis',ofname='figures/gseadotplot/'+i+j+'.pdf')
                df_tmp = enr.results
                df_tmp.index = df_tmp['Term'].values
                df_tmp = df_tmp[['Adjusted P-value']]
                df_tmp.columns = [i+'_'+j]
                if df_all is None:
                    df_all = df_tmp
                else:
                    df_all = pd.concat([df_all,df_tmp],axis=1)
                    
    df_all = df_all.loc[df_all.min(axis=1)<0.05]
    df_all = df_all.fillna(1)
    df_all = -np.log10(df_all)
    df_all.to_csv('gseapy_up.csv')
    df_all = None
    for i in adata.obs[label].cat.categories:
        j = 'down'
        gl = pd.read_csv('genelist/'+i+j+'.csv',index_col=0).index.values
        if gl.shape[0]>0:
            enr = gp.enrichr(gene_list=list(gl), # or "./tests/data/gene_list.txt",
                         gene_sets=['WikiPathways_2019_Mouse'],
                         organism='Mus musculus', # don't forget to set organism to the one you desired! e.g. Yeast
                         outdir=None, # don't write to disk
                        )
            if not enr.results.empty:
                if enr.results['Adjusted P-value'].min() < 0.05:
                    dotplot(enr.results,column = 'Adjusted P-value',size=50,cmap='viridis',ofname='figures/gseadotplot/'+i+j+'.pdf')
                df_tmp = enr.results
                df_tmp.index = df_tmp['Term'].values
                df_tmp = df_tmp[['Adjusted P-value']]
                df_tmp.columns = [i+'_'+j]
                if df_all is None:
                    df_all = df_tmp
                else:
                    df_all = pd.concat([df_all,df_tmp],axis=1)
                    
    df_all = df_all.loc[df_all.min(axis=1)<0.05]
    df_all = df_all.fillna(1)
    df_all = -np.log10(df_all)
    df_all.to_csv('gseapy_down.csv')
    return
    
    
def annotate(adata,label,names):
    new_cluster_names = np.array(names)
    celltype_label = new_cluster_names[np.array(list(map(int, adata.obs['leiden'].tolist())))]
    adata.obs[label] = celltype_label
    
def extract_bac(adata_,label,umi_thres = 0):
    adata = adata_[:,adata_.var['species']!='Host'].copy()
    adata = adata[adata.obs['bac_umi']>umi_thres]
    adata = adata[:,adata.var['biotype']=='protein_coding']
    df = pd.DataFrame(adata.X.toarray(),index=adata.obs_names)
    df[label] = adata.obs[label].values.copy()
    adata = sc.AnnData(df.groupby(label).sum())
    adata.obs[label] = adata.obs_names.values.copy()
    return adata

def extract_bac_species(adata_,species,umi_thres = 0):
    adata = adata_[:,adata_.var['species']==species].copy()
    adata = adata[:,adata.var['biotype']=='protein_coding']
    #adata = adata[:,adata.var['biotype']=='protein_coding']
    
    adata = adata[adata.obs[species]>umi_thres]
    sc.pp.filter_genes(adata,min_cells=1)
    #sc.pp.filter_cells(adata,min_genes=5)
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)
    #sc.pp.pca(adata)
    return adata

def pie_chart(adata,lower_thres=0.005,save='piechart.pdf',exclude=None,include=None):
    df = adata.obsm['species_abundance'].copy()
    df_tmp = adata.var[['name','species']].copy().groupby('species').first()
    df.columns = df_tmp['name'][df.columns]
    df = df.loc[:,(df.sum(axis=0)>100)]
    df = np.sqrt(df)
    if exclude is not None:
        df = df.loc[:,~df.columns.isin(exclude)]
    if include is not None:
        df = df.loc[:,df.columns.isin(include)]
    labels = df.columns
    sizes = np.sqrt(adata.obs['bac_umi'].values) * 5
    sizes = np.clip(sizes / (20 * np.max(sizes)),lower_thres,0.03) - lower_thres
    fig = plt.figure(figsize=(6, 6))
    for i in range(adata.shape[0]):
        if sizes[i]>lower_thres and df.sum(axis=1)[i] > 0:
            ax1 = fig.add_axes([(adata.obsm['spatial'][i,0]/50) - 0.5 * (sizes[i]), ((50-adata.obsm['spatial'][i,1])/50) - 0.5 * (sizes[i]), sizes[i], sizes[i]])  # 位置1
            ax1.pie(df.values[i],colors=palette_20_)

    fig.legend(labels, loc="center left", bbox_to_anchor=(1.05, 0.5))
    plt.savefig(save)
    return

def circle_chart(adata,save='figures/circlechart.pdf',include=None):
    fig, axes = plt.subplots(math.ceil(adata.obsm['species_abundance'].shape[1]/4),4,figsize=(25,25))
    j = 0
    k = 0
    for i in adata.obsm['species_abundance'].columns:
        df = adata.obs[[i+'_mRNA',i+'_rRNA',i+'_tRNA']].values.sum(axis=0)
        labels = ['mRNA','rRNA','tRNA']
        axes[j,k].pie(df, labels=labels, startangle=90, wedgeprops=dict(width=0.3), autopct=lambda p: '{:.1f}%'.format(p) if p > 1 else '')
        axes[j,k].text(0, 0, adata.var['name'][adata.var['species']==i][0], ha='center', va='center')
        k = k + 1
        if k == 4:
            j = j + 1
            k = 0
    
    plt.tight_layout()
    plt.savefig(save)
    return

def umi_barplot(adata,stacked=False,thres=0.05,save='figures/umi_pieplot.pdf'):
    df = adata.obsm['species_abundance'].sum(axis=0)
    df_tmp = adata.var[['name','species']].copy().groupby('species').first()
    df.index = df_tmp['name'][df.index]
    df = df[df.index!='Host']
    if stacked:
        df = df.sort_values(ascending=False)
        df.plot(kind='bar',stacked=False)
        plt.savefig('figures/umi_barplot.pdf')
    else:
        df_ = df/df.sum()
        df_ = df_[df_>thres]
        df_ = pd.DataFrame(df_).T
        df_['Others'] = 1 - df_.values.sum()
        df_.T[0].plot(kind='pie',stacked=stacked,colors=palette_20_,fontsize=5, autopct='%1.1f%%')
        plt.savefig(save)
    return
    
def var_barplot(adata,label='biotype', stacked=False,thres=0.05,save='figures/'):
    df = pd.DataFrame(adata[:,adata.var['species']!='Host'].X.T.sum(axis=1))
    df[label] = adata.var[label][adata.var['species']!='Host'].values.astype(str).copy()
    df = df.groupby(label).sum()
    #df = df[df.index!='Host']
    if stacked:
        df_ = df/df.sum()
        df_ = df_.loc[df_[0]>thres]
        df_ = pd.DataFrame(df_).T
        if df_.values.sum() != 1:
            df_['Others'] = 1 - df_.values.sum()
        ax = df_.plot(kind='bar',stacked=stacked,figsize=(2,4))
        for i in ax.patches:
            x, height = i.get_x(), i.get_height()
            y = i.get_y()
            if height > 0:
                ax.text(0, y + height / 2, f'{height*100:.1f}%', ha='center')
    else:
        df_ = df/df.sum()
        df_ = df_[df_>thres]
        df_ = pd.DataFrame(df_).T
        df_['Others'] = 1 - df_.values.sum()
        df_.T.dropna()[0].plot(kind='pie', wedgeprops=dict(width=0.3), autopct=lambda p: '{:.1f}%'.format(p) if p > 1 else '')
    plt.axis('off')
    plt.legend(loc="center left", bbox_to_anchor=(1.05, 0.5))
    plt.savefig(save+label+'_pieplot.pdf')
    return


    
def correlationmap(adata):
    mRNA_mt = adata.obsm['species_abundance'].copy()
    rRNA_mt = adata.obsm['species_abundance'].copy()
    for i in adata.var['species']:
        if i != 'Host':
            mRNA_mt[i] = adata.obs[i+'_mRNA'].values
            rRNA_mt[i] = adata.obs[i+'_rRNA'].values
    mRNA_mt = mRNA_mt.loc[:,mRNA_mt.sum()>0]
    rRNA_mt = rRNA_mt.loc[:,rRNA_mt.sum()>0]
    mRNA_mt.columns = mRNA_mt.columns + '_mRNA'
    rRNA_mt.columns = rRNA_mt.columns + '_rRNA'    
    df_concat = pd.concat([mRNA_mt, rRNA_mt], axis=1)
    corr_matrix = df_concat.corr()
    mRNA_mRNA = corr_matrix.iloc[0:mRNA_mt.shape[1],0:mRNA_mt.shape[1]]
    mRNA_rRNA = corr_matrix.iloc[0:mRNA_mt.shape[1],mRNA_mt.shape[1]:]
    rRNA_rRNA = corr_matrix.iloc[mRNA_mt.shape[1]:,mRNA_mt.shape[1]:]
    orde = correlation_plot(rRNA_rRNA,None,'rRNA')
    correlation_plot(mRNA_mRNA,None,'mRNA')
    return corr_matrix

def correlation_plot(corr_matrix,orde,name):
    import seaborn as sns
    if orde is None:
        a = sns.clustermap(corr_matrix,figsize=(14, 14),cmap='coolwarm',vmax=1,vmin=-1)
        a.savefig('figures/'+name+'_colocalization.pdf')
        return a.dendrogram_row.reordered_ind
    else:
        a = sns.clustermap(corr_matrix.iloc[orde,orde],row_cluster=False,figsize=(14, 14),col_cluster=False,cmap='coolwarm',vmax=1,vmin=-1)
        a.savefig('figures/'+name+'_colocalization.pdf')
        return
    

In [ ]:
# code execution
adata_all = vis_comparison(['CMM'],'CMS')

In [ ]:
# generate plots
plt.rcParams["font.family"] = 'Arial'
plt.rcParams['figure.dpi'] = 250
plt.rcParams["font.size"] = 18
sc.tl.rank_genes_groups(adata_all, 'leiden',use_raw=False)
sc.pl.rank_genes_groups_dotplot(adata_all,n_genes=10,dendrogram=False,save='leidenall.pdf',vmax=3)

In [ ]:
# code execution
adata = preprocessing('CMM', 'CMM.jpg', res = 80.5, mode = 'csv')
adata.obs['leiden'] = adata_all[adata_all.obs['batch']=='CMM'].obs['leiden'].values.copy()
annotate(adata_all,'celltype',['Epithelial','Lamina Propria / Crypt','Smooth muscle','Epithelial','Lamina Propria / Crypt','Others (neural-like)','Others (neural-like)','Others (neural-like)','Others (neural-like)','Lamina Propria / Crypt'])
adata.obs['celltype'] = adata_all[adata_all.obs['batch']=='CMM'].obs['celltype'].values.copy()

adata.uns['celltype_colors'] = [ '#41ab5d', '#08519c', '#ae017e', '#969696']
sc.pl.spatial(adata,color=['leiden','celltype','bac_umi'],vmax='p99',wspace=0.6,save='cmm_ct.pdf')

In [ ]:
# code execution
adata_cms = preprocessing('CMS', 'CMS.jpg', res = 80, mode = 'csv')
adata_cms.obs['leiden'] = adata_all[adata_all.obs['batch']=='GF'].obs['leiden'].values.copy()
adata_cms.obs['celltype'] = adata_all[adata_all.obs['batch']=='GF'].obs['celltype'].values.copy()

adata_cms.uns['celltype_colors'] = [ '#41ab5d', '#08519c', '#ae017e', '#969696']
sc.pl.spatial(adata_cms,color=['leiden','celltype','bac_umi'],vmax='p99',wspace=0.6,save='cms_ct.pdf')

In [ ]:
# code execution
sc.tl.rank_genes_groups(adata_all, 'celltype',use_raw=False)
sc.pl.rank_genes_groups_dotplot(adata_all,n_genes=10,dendrogram=False,save='celltypeall.pdf',vmax=3)

In [ ]:
# code execution
adata_raw = preprocessing('CMM', 'CMM.jpg', res = 80, mode = 'tsv')
#umi_barplot(adata_raw,stacked=True)

In [ ]:
# code execution
adata_cmsraw = preprocessing('CMS', 'CMS.jpg', res = 80, mode = 'csv')
#umi_barplot(adata_cmsraw,stacked=True)

In [ ]:
# generate plots
df_ = umi_barplot(adata_raw,thres=0.002)

In [ ]:
# generate plots
df_ = umi_barplot(adata_cmsraw,thres=0.0025)

In [ ]:
# generate plots
df_ = var_barplot(adata_raw,thres=1e-2)

In [ ]:
# generate plots
df_ = var_barplot(adata_cmsraw,thres=1e-2)

In [ ]:
# code execution
sc.pl.umap(adata_all,color=['condition','celltype'],wspace=0.5,save='celltype.pdf')

In [ ]:
# import necessary libraries
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

def make_colormap(seq):
    seq = [(None,) * 3, 0.0] + list(seq) + [1.0, (None,) * 3]
    cdict = {'red': [], 'green': [], 'blue': []}
    for i, item in enumerate(seq):
        if isinstance(item, float):
            r1, g1, b1 = seq[i - 1]
            r2, g2, b2 = seq[i + 1]
            cdict['red'].append([item, r1, r2])
            cdict['green'].append([item, g1, g2])
            cdict['blue'].append([item, b1, b2])
    return mcolors.LinearSegmentedColormap('CustomMap', cdict)

c = mcolors.ColorConverter().to_rgb
cmap = make_colormap([c("lightgrey"), c("blue")])

In [ ]:
# code execution
sc.pl.spatial(sq.pl.extract(adata, "species_abundance"),color=adata.obsm['species_abundance'].columns,cmap=cmap,vmax='p99',save='cmmbacumi_summary.pdf')

In [ ]:
# code execution
sc.pl.spatial(sq.pl.extract(adata_cms, "species_abundance"),color=adata_cms.obsm['species_abundance'].columns,cmap=cmap,vmax='p99',save='cmsbacumi_summary.pdf')

In [ ]:
# code execution
sc.pl.spatial(adata_cms,color=['Bovatus_mRNA','Bovatus_rRNA'],cmap=cmap,vmax='p99',save='cms_bovatus.pdf')

In [ ]:
# code execution
sc.pl.spatial(adata,color=['Bovatus_mRNA','Bovatus_rRNA'],cmap=cmap,vmax='p99',save='cmm_bovatus.pdf')

In [ ]:
# code execution
sc.pl.spatial(adata,color=['GOZ73_mRNA','GOZ73_rRNA'],cmap=cmap,vmax='p99',save='cmm_GOZ73.pdf')

In [ ]:
# code execution
pie_chart(adata,save='figures/cmm_piechart.pdf')

In [ ]:
# code execution
pie_chart(adata_cms,save='figures/cms_piechart.pdf')

In [ ]:
# code execution
adata_raw.obs['celltype'] = adata.obs['celltype'].values.copy()
adata_raw.uns['celltype_colors'] = adata.uns['celltype_colors'].copy()
sc.pl.violin(adata_raw,'bac_umi','celltype',rotation=15,save='cmm_umiviolin.pdf')

In [ ]:
# code execution
adata_cmsraw.obs['celltype'] = adata_cms.obs['celltype'].values.copy()
adata_cmsraw.uns['celltype_colors'] = adata_cms.uns['celltype_colors'].copy()
sc.pl.violin(adata_cmsraw,'bac_umi','celltype',rotation=15,save='cms_umiviolin.pdf')

In [ ]:
# code execution
x = correlationmap(adata_cms[(adata_cms.obs['bac_umi']>200)])

In [ ]:
#circle_chart(adata_cmmraw)
circle_chart(adata_cmsraw)